# Cars: Getting Started

In [347]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
sns.set_theme()

# Read in data

The dataset, `cars.csv`, contains cleaned and processed data. If you want to make changes, the notebook, `materialize.ipynb`, creates the data from the raw source datsets. 

In [348]:
cars = pd.read_csv('data/cars.csv') # this reads the *balanced* dataset (i.e. J = 40 products per market always)
# cars = pd.read_excel('cars.xlsx') # this reads the *unbalanced* dataset (i.e. J varies over time)
lbl_vars = pd.read_csv('data/labels_variables.csv', index_col=0)
lbl_vals = pd.read_stata('data/cars.dta', iterator=True).value_labels() # the values that variables take (not relevant for all )

## Overview of the dataset

In [349]:
tab = cars.mean(numeric_only=True).apply(lambda x: f'{x:.2f}').to_frame('Mean').join(lbl_vars)
tab

,Mean,label
ye,84.50,year (=first dimension of panel)
ma,3.00,market (=second dimension of panel)
co,207.50,model code (=third dimension of panel)
zcode,177.76,alternative model code (predecessors and succe...
brd,16.79,brand code
org,2.72,"origin code (demand side, country with which c..."
loc,5.17,"location code (production side, country where ..."
cla,2.30,class or segment code
home,0.32,domestic car dummy (appropriate interaction of...
frm,14.50,firm code


# Set up for analysis

## Price variables 
Can be either price (`pr`), price-to-income (`princ`), or log price (`logp`, created below).

In [ ]:
price_var = 'princ'
cars['logp'] = np.log(cars[price_var])

Index(['ye', 'ma', 'co', 'zcode', 'brd', 'type', 'brand', 'model', 'org',
       'loc', 'cla', 'home', 'frm', 'qu', 'cy', 'hp', 'we', 'pl', 'do', 'le',
       'wi', 'he', 'li1', 'li2', 'li3', 'li', 'sp', 'ac', 'pr', 'princ',
       'eurpr', 'exppr', 'avexr', 'avdexr', 'avcpr', 'avppr', 'avdcpr',
       'avdppr', 'xexr', 'tax', 'pop', 'ngdp', 'rgdp', 'engdp', 'ergdp',
       'engdpc', 'ergdpc', 'inc', 'logp'],
      dtype='object')

## Market share

In [ ]:
# total quantity of cars sold in market-year (ma, ye)
cars['qu_tot'] = cars.groupby(['ma', 'ye'])['qu'].transform('sum')
# TODO: marketsize er en antagelse (her antager vi at hvert tredje person i et land er i bilmarkedet)
cars['market_size'] = cars['pop'] / 3
cars['s'] = cars['qu'] / cars['market_size']

# compute the share of the outside good (will be useful for the demand inversion)
cars['s0'] = 1.0 - cars.groupby(['ma', 'ye'])['s'].transform('sum')
print(f'Outside share is from {cars.s0.min():.1%} to {cars.s0.max():.1%}')

assert (cars["s"] > 0).all() and (cars["s0"] > 0).all()

In [353]:
cars.groupby(['ma'])['s'].describe().rename(index=lbl_vals['market']).style.format('{:.3f}')

,count,mean,std,min,25%,50%,75%,max
ma,,,,,,,,
Belgium,1200.000,0.002,0.001,0.000,0.001,0.002,0.003,0.019
France,1200.000,0.002,0.002,0.000,0.001,0.001,0.003,0.017
Germany,1200.000,0.002,0.003,0.000,0.001,0.001,0.003,0.019
Italy,1200.000,0.002,0.003,0.000,0.001,0.001,0.002,0.023
UK,1200.000,0.002,0.002,0.000,0.001,0.001,0.002,0.010


## Regressions

In [354]:
cars.type.value_counts().sort_values()

type
ford scorpio/granada      1
alfa 166                  1
daewoo matiz/M            1
mazda xedos               1
innocenti 650             1
                       ... 
opel kadett             108
volkswagen polo         115
ford fiesta             116
volkswagen golf         126
ford escort             143
Name: count, Length: 323, dtype: int64

In [355]:
from linearmodels.iv import IV2SLS

In [ ]:
# Simpel logit (delta_jt = np.log(s_jt) - np.log(s_0t))
cars['market'] = cars['ye'].astype(str) + '_' + cars['ma'].astype(str)

cars['delta'] = np.log(cars['s']) - np.log(cars['s0'])

cars['cla_nest_total_share'] = cars.groupby(['ye','ma','cla'])['s'].transform('sum')
cars['within_cla_nest_share'] = cars['s'] / cars['cla_nest_total_share'] 


cars['country_nest_total_share'] = cars.groupby(['ye','ma'])['s'].transform('sum')

formulas = [
    'delta ~ 1 + logp',
    'delta ~ 1 + logp + we + li + C(brd)',
    'delta ~ 1 + logp + we + li + C(co)',
    'delta ~ 1 + logp + we + li + C(co) + home',
    'delta ~ 1 + logp + we + li + C(co) + C(market)',
    'delta ~ 1 + logp                                     + np.log(within_cla_nest_share)',
    'delta ~ 1 + logp + we + li                           + np.log(within_cla_nest_share)',
    'delta ~ 1 + logp + we + li + C(brd)                  + np.log(within_cla_nest_share)',
    'delta ~ 1 + logp + we + li + C(brd) + C(market)      + np.log(within_cla_nest_share)',
]

for formula in formulas:
    model_simple_logit = IV2SLS.from_formula(formula, data=cars).fit(cov_type="robust")
    # display(model_simple_logit.summary)

    print(f"model:                    {formula}")
    print(f"Price coefficient (logp): {model_simple_logit.params['logp']:.4f}")
    print(f"Std. error:               {model_simple_logit.std_errors['logp']:.4f}")
    print(f"t-stat:                   {model_simple_logit.tstats['logp']:.2f}")
    print("\n")

model:                    delta ~ 1 + logp
Price coefficient (logp): -0.6162
Std. error:               0.0273
t-stat:                   -22.59


model:                    delta ~ 1 + logp + we + li + C(brd)
Price coefficient (logp): -0.9596
Std. error:               0.0495
t-stat:                   -19.39


model:                    delta ~ 1 + logp + we + li + C(co)
Price coefficient (logp): -0.9757
Std. error:               0.0555
t-stat:                   -17.56


model:                    delta ~ 1 + logp + we + li + C(co) + home
Price coefficient (logp): -0.8165
Std. error:               0.0436
t-stat:                   -18.71


model:                    delta ~ 1 + logp + we + li + C(co) + C(market)
Price coefficient (logp): -0.8097
Std. error:               0.1419
t-stat:                   -5.71


model:                    delta ~ 1 + logp                                     + np.log(within_cla_nest_share)
Price coefficient (logp): -1.2668
Std. error:               0.0216
t-stat

## regression table output

In [357]:
def latex_regression_table(model, filename="simple_logit_table.tex"):

    labels = {
        "Intercept": "Constant",
        "logp": r"$\log(p_{jt})$",
        "we": "Weight",
        "li": "Size",
    }

    lines = [
        r"\begin{table}[htbp]",
        r"    \centering",
        r"    \caption{Simple Logit Demand Estimates}",
        r"    \label{tab:simple_logit}",
        r"    ",
        r"    \begin{threeparttable}",
        r"        \begin{tabular}{lc}",
        r"            \toprule",
        r"            & \textbf{Simple Logit} \\",
        r"            \midrule",
    ]

    for var in model.params.index:

        b = model.params[var]
        se = model.std_errors[var]
        p = model.pvalues[var]

        if p < 0.01:
            stars = "***"
        elif p < 0.05:
            stars = "**"
        elif p < 0.10:
            stars = "*"
        else:
            stars = ""

        name = labels.get(var, var)

        lines += [
            f"            {name} & {b:.3f}{stars} \\\\",
            f"                 & ({se:.3f}) \\\\[0.3em]",
        ]

    lines += [
        r"            \midrule",
        f"            Observations & {model.nobs:,} \\\\",
        f"            $R^2$ & {model.rsquared:.3f} \\\\",
        r"            \bottomrule",
        r"        \end{tabular}",
        r"        ",
        r"        \begin{tablenotes}[flushleft]",
        r"            \footnotesize",
        r"            \item \textit{Notes:} The dependent variable is",
        r"            $\delta_{jt} = \log(s_{jt}) - \log(s_{0t})$.",
        r"            Robust standard errors are reported in parentheses.",
        r"            *** $p<0.01$, ** $p<0.05$, * $p<0.10$.",
        r"        \end{tablenotes}",
        r"    \end{threeparttable}",
        r"\end{table}",
    ]

    latex = "\n".join(lines)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(latex)

    print(latex)
latex_regression_table(model_simple_logit)

\begin{table}[htbp]
    \centering
    \caption{Simple Logit Demand Estimates}
    \label{tab:simple_logit}
    
    \begin{threeparttable}
        \begin{tabular}{lc}
            \toprule
            & \textbf{Simple Logit} \\
            \midrule
            Constant & -6.611*** \\
                 & (0.165) \\[0.3em]
            $\log(p_{jt})$ & -0.637*** \\
                 & (0.085) \\[0.3em]
            Weight & -0.001*** \\
                 & (0.000) \\[0.3em]
            Size & -0.026** \\
                 & (0.011) \\[0.3em]
            C(brd)[T.2] & 0.311*** \\
                 & (0.071) \\[0.3em]
            C(brd)[T.3] & -0.283*** \\
                 & (0.080) \\[0.3em]
            C(brd)[T.4] & 0.089 \\
                 & (0.068) \\[0.3em]
            C(brd)[T.7] & 0.187** \\
                 & (0.073) \\[0.3em]
            C(brd)[T.8] & 0.497*** \\
                 & (0.067) \\[0.3em]
            C(brd)[T.9] & -0.120 \\
                 & (0.074) \\[0.3em]
            C(b

In [ ]:
"""Level-price demand: OLS, IV logit and demand-only BLP.
"""

import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import pyblp


ROOT = Path(__file__).resolve().parent
OUT = ROOT / "output_simple_demand"
OUT.mkdir(exist_ok=True)

pyblp.options.verbose = False


# ------------------------------------------------------------------
# 1. Load data and define prices, characteristics and market shares
# ------------------------------------------------------------------

d = pd.read_csv(ROOT / "input" / "cars.csv")

d = (
    d.sort_values(["ma", "ye", "co"])
    .reset_index(drop=True)
)

d["market_ids"] = (
    d["ma"].astype(str) + "_" + d["ye"].astype(str)
)

d["firm_ids"] = d["frm"]
d["clustering_ids"] = d["co"]

# Price relative to annual GDP per capita.
d["prices"] = d["princ"]

# One potential annual purchase opportunity per household.
d["market_size"] = d["pop"] / 2.5
d["shares"] = d["qu"] / d["market_size"]

d["outside_share"] = (
    1 - d.groupby("market_ids")["shares"].transform("sum")
)

d["weight"] = d["we"] / 1000
d["fuel"] = d["li"]
d["power"] = d["hp"] / d["weight"] / 100
d["size"] = d["le"] * d["wi"] / 10000

controls = ["weight", "fuel", "home", "power", "size"]

required = controls + ["prices", "shares", "outside_share"]

assert d[required].notna().all().all()
assert (d[["prices", "shares", "outside_share"]] > 0).all().all()

print(f"Observations: {len(d):,}")
print("Markets:", d["market_ids"].nunique())
print(
    "Mean included-car share:",
    d.groupby("market_ids")["shares"].sum().mean(),
)


# ------------------------------------------------------------------
# 2. Invert homogeneous-logit demand
#
# delta = x'beta + brand FE + market FE - alpha*price + xi
# ------------------------------------------------------------------

d["delta"] = (
    np.log(d["shares"])
    - np.log(d["outside_share"])
)

print("\nFirst five inverted utilities:")
print(d["delta"].head().to_numpy())


# ------------------------------------------------------------------
# 3. Construct four rival-only quadratic differentiation instruments
#
# For each characteristic:
# Z_j = sum over rival products k of (x_k - x_j)^2
# ------------------------------------------------------------------

distances = pyblp.build_differentiation_instruments(
    pyblp.Formulation("0 + weight + fuel + power + size"),
    d,
    version="quadratic",
)

# PyBLP returns four own-firm columns, then four rival columns.
Z = distances[:, 4:]

# Match the scaling used in the earlier estimates.
Z = Z / Z.std(axis=0)

instrument_names = [
    "rival_weight",
    "rival_fuel",
    "rival_power",
    "rival_size",
]

d[instrument_names] = Z

# PyBLP recognises excluded instruments by these column names.
for k in range(4):
    d[f"demand_instruments{k}"] = Z[:, k]


# ------------------------------------------------------------------
# 4. OLS: one formula, with fixed effects included directly
# ------------------------------------------------------------------

ols = smf.ols(
    formula=(
        "delta ~ prices "
        "+ weight + fuel + home + power + size "
        "+ C(market_ids) + C(brd)"
    ),
    data=d,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": d["co"],
        # Match the earlier uncorrected clustered covariance.
        "use_correction": False,
    },
)


# ------------------------------------------------------------------
# 5. IV: one formula; linearmodels performs both stages internally
# ------------------------------------------------------------------

iv = IV2SLS.from_formula(
    formula=(
        "delta ~ 1 "
        "+ weight + fuel + home + power + size "
        "+ C(market_ids) + C(brd) "
        "+ [prices ~ rival_weight + rival_fuel "
        "+ rival_power + rival_size]"
    ),
    data=d,
).fit(
    cov_type="clustered",
    clusters=d["co"],
    debiased=False,
)

first_stage = iv.first_stage.diagnostics

print("\nIV first-stage diagnostics:")
print(first_stage.to_string())

print(
    "Clustered first-stage Wald / 4:",
    first_stage.iloc[0]["f.stat"] / 4,
)

print("\nPrice-exogeneity test, conditional on valid instruments:")
print(iv.wooldridge_regression)

# These diagnostics do not establish instrument exclusion.


# ------------------------------------------------------------------
# 6. Quadrature for the BLP consumer integral
# ------------------------------------------------------------------

quadrature = pyblp.Integration("product", size=40)

rule = pyblp.build_integration(
    quadrature,
    dimensions=1,
)

normal_nodes = rule["nodes"][:, 0]
normal_weights = rule["weights"].ravel()

market_ids = d["market_ids"].drop_duplicates().to_numpy()

agents = pd.DataFrame({
    "market_ids": np.repeat(market_ids, len(normal_nodes)),
    "nodes0": np.tile(normal_nodes, len(market_ids)),
    "weights": np.tile(normal_weights, len(market_ids)),
    "one": 1.0,
})

print("\nQuadrature points per market:", len(normal_nodes))
print("Quadrature weights sum to:", normal_weights.sum())


# ------------------------------------------------------------------
# 7. Demand-only BLP
#
# Utility:
# x'beta + brand FE + market FE + xi
# - exp(mu + sigma*nu_i)*price + epsilon
#
# PyBLP handles share inversion, FE absorption and two-step GMM.
# ------------------------------------------------------------------

x1 = pyblp.Formulation(
    "0 + weight + fuel + home + power + size",
    absorb="C(market_ids) + C(brd)",
)

# Positive lognormal sensitivity multiplied by negative price.
x2 = pyblp.Formulation("0 + I(-prices)")

problem = pyblp.Problem(
    product_formulations=(x1, x2),
    product_data=d,
    agent_formulation=pyblp.Formulation("0 + one"),
    agent_data=agents,
    rc_types=["log"],
)

blp = problem.solve(
    sigma=np.array([[0.25]]),
    pi=np.array([[
        np.log(max(-iv.params["prices"], 2.0))
    ]]),
    sigma_bounds=(0, 1.5),
    pi_bounds=(-3, 5),
    method="2s",
    W_type="clustered",
    se_type="clustered",
    optimization=pyblp.Optimization(
        "l-bfgs-b",
        {
            "gtol": 1e-7,
            "ftol": 1e-12,
            "maxiter": 220,
            "maxls": 40,
        },
    ),
    iteration=pyblp.Iteration(
        "squarem",
        {
            "atol": 1e-12,
            "max_evaluations": 4000,
        },
    ),
    error_behavior="raise",
)

mu = blp.pi.item()
sigma = blp.sigma.item()
mean_alpha = np.exp(mu + sigma**2 / 2)

share_error = np.max(np.abs(
    blp.compute_shares().ravel()
    - d["shares"].to_numpy()
))

print("\nBLP converged:", blp.converged)
print("BLP maximum share error:", share_error)
print("BLP projected gradient:", blp.projected_gradient_norm)

assert blp.converged
assert share_error < 1e-9
assert blp.projected_gradient_norm < 1e-4
assert not blp.clipped_shares.any()

# Reject solutions hitting the artificial upper/location bounds.
assert -2.999 < mu < 4.999
assert sigma < 1.499


# ------------------------------------------------------------------
# 8. Display compact coefficient tables
# ------------------------------------------------------------------

terms = controls + ["prices"]

coefficients = pd.DataFrame({
    "OLS": ols.params.reindex(terms),
    "OLS_SE": ols.bse.reindex(terms),
    "IV": iv.params.reindex(terms),
    "IV_SE": iv.std_errors.reindex(terms),
    "BLP": np.r_[blp.beta.ravel(), np.nan],
    "BLP_SE": np.r_[blp.beta_se.ravel(), np.nan],
}, index=terms)

price_distribution = pd.DataFrame({
    "estimate": [mu, sigma, mean_alpha],
    "SE": [
        blp.pi_se.item(),
        blp.sigma_se.item(),
        np.nan,
    ],
}, index=["mu", "sigma", "mean_alpha"])

print("\nDemand coefficients:")
print(coefficients.round(6).to_string())

print("\nBLP price-sensitivity distribution:")
print(price_distribution.round(6).to_string())

print("\nMean positive price sensitivity:")
print("OLS:", -ols.params["prices"])
print("IV: ", -iv.params["prices"])
print("BLP:", mean_alpha)


# ------------------------------------------------------------------
# 9. Compare with the earlier rival-only estimates
#
# These reference checks apply to the supplied cars.csv and this exact
# specification. They intentionally fail if the specification changes.
# ------------------------------------------------------------------

reference_coefficients = np.array([
    # OLS       OLS_SE     IV        IV_SE     BLP       BLP_SE
    [-0.024795, 0.313234,  1.504519, 0.583521, 2.143330, 1.279083],
    [-0.025556, 0.022160, -0.027740, 0.024629, -0.039343, 0.025713],
    [ 1.221877, 0.048788,  1.210898, 0.048098, 1.225778, 0.047615],
    [-0.618647, 0.276553,  0.564240, 0.520751, 1.046982, 1.070972],
    [ 0.089845, 0.061539,  0.103722, 0.065178, 0.169724, 0.122406],
    [-1.108489, 0.178731, -2.822323, 0.645268, np.nan, np.nan],
])

np.testing.assert_allclose(
    coefficients.to_numpy(),
    reference_coefficients,
    rtol=0,
    atol=5e-4,
    equal_nan=True,
)

reference_distribution = np.array([
    [1.511499, 0.905141],
    [0.259824, 0.303222],
    [4.689156, np.nan],
])

np.testing.assert_allclose(
    price_distribution.to_numpy(),
    reference_distribution,
    rtol=0,
    atol=5e-4,
    equal_nan=True,
)

np.testing.assert_allclose(
    [
        first_stage.iloc[0]["partial.rsquared"],
        first_stage.iloc[0]["f.stat"],
    ],
    [0.116123270958, 91.6304197068],
    rtol=0,
    atol=1e-4,
)

print("\nPASS: estimates match the earlier rival-only results.")


# ------------------------------------------------------------------
# 10. Save results for inspection and the later merger calculation
# ------------------------------------------------------------------

coefficients.to_csv(
    OUT / "coefficients.csv",
    index_label="parameter",
)

price_distribution.to_csv(
    OUT / "blp_price_distribution.csv",
    index_label="parameter",
)

first_stage.to_csv(OUT / "first_stage.csv")

d.to_csv(OUT / "prepared.csv", index=False)

with (OUT / "blp_results.pkl").open("wb") as file:
    pickle.dump(blp, file)

print("Saved results to:", OUT)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\caspe\\OneDrive - University of Copenhagen\\Kandidat\\7_semester\\AIO\\projects\\P1_EUcars\\data\\input\\cars.csv'